# Part D — Scikit-learn Machine Learning

Problems 11–15 use the UCI Student Performance dataset. Run Part C first if `student-mat.csv` is not present. The final-grade target is `G3` (0–20).

In [ ]:
from pathlib import Path
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, precision_score, recall_score, f1_score,
                             ConfusionMatrixDisplay, classification_report)
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer

DATA_FILE = Path("student-mat.csv")
if not DATA_FILE.exists():
    urllib.request.urlretrieve("https://archive.ics.uci.edu/ml/machine-learning-databases/00320/student-mat.csv", DATA_FILE)
df = pd.read_csv(DATA_FILE, sep=";")
print("Dataset shape:", df.shape)

## Problem 11. Regression

In [ ]:
regression_features = ["studytime", "failures", "absences", "G1", "G2"]
X_reg = df[regression_features]
y_reg = df["G3"]
X_train, X_test, y_train, y_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

regression_models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(max_depth=5, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=150, random_state=42)
}
regression_results = []
for name, model in regression_models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    regression_results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, predictions),
        "MSE": mean_squared_error(y_test, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_test, predictions)),
        "R2": r2_score(y_test, predictions)
    })
regression_table = pd.DataFrame(regression_results).sort_values("RMSE")
display(regression_table.round(3))
best_regression_name = regression_table.iloc[0]["Model"]
print("Best model on this test split:", best_regression_name,
      "because it has the lowest RMSE among these runs.")

## Problem 12. Classification

In [ ]:
# Predict pass/fail. Grade columns are excluded from the input features.
classification_features = [
    "age", " Medu", "Fedu", "traveltime", "studytime", "failures",
    "famrel", "freetime", "goout", "Dalc", "Walc", "health", "absences"
]
classification_features = [c.strip() for c in classification_features]
# The source dataset uses 'Medu' and 'Fedu' without leading spaces.
classification_features = [c for c in classification_features if c in df.columns]
X_cls = df[classification_features]
y_cls = (df["G3"] >= 10).astype(int)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)
classification_models = {
    "Logistic Regression": Pipeline([("scale", StandardScaler()), ("model", LogisticRegression(max_iter=1000))]),
    "K-Nearest Neighbors": Pipeline([("scale", StandardScaler()), ("model", KNeighborsClassifier(n_neighbors=5))]),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=150, random_state=42)
}
classification_results = []
fitted_classifiers = {}
for name, model in classification_models.items():
    model.fit(Xc_train, yc_train)
    predictions = model.predict(Xc_test)
    fitted_classifiers[name] = model
    classification_results.append({
        "Model": name,
        "Accuracy": accuracy_score(yc_test, predictions),
        "Precision": precision_score(yc_test, predictions, zero_division=0),
        "Recall": recall_score(yc_test, predictions, zero_division=0),
        "F1-score": f1_score(yc_test, predictions, zero_division=0)
    })
classification_table = pd.DataFrame(classification_results).sort_values("F1-score", ascending=False)
display(classification_table.round(3))

chosen_classifier_name = classification_table.iloc[0]["Model"]
chosen_classifier = fitted_classifiers[chosen_classifier_name]
cls_predictions = chosen_classifier.predict(Xc_test)
print("Classification report for", chosen_classifier_name)
print(classification_report(yc_test, cls_predictions, target_names=["Fail (<10)", "Pass (>=10)"], zero_division=0))
ConfusionMatrixDisplay.from_predictions(
    yc_test, cls_predictions, display_labels=["Fail", "Pass"], cmap="Blues"
)
plt.title(f"Confusion matrix — {chosen_classifier_name}")
plt.tight_layout()
plt.savefig("classification_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("False positive: predicted pass when the student actually failed.")
print("False negative: predicted fail when the student actually passed.")

## Problem 13. Clustering

In [ ]:
cluster_features = ["studytime", "absences", "G3"]
cluster_data = StandardScaler().fit_transform(df[cluster_features])
inertias = []
k_values = range(1, 8)
for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(cluster_data)
    inertias.append(model.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(list(k_values), inertias, marker="o")
plt.xlabel("Number of clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow method for K-Means")
plt.tight_layout()
plt.savefig("kmeans_elbow.png", dpi=150, bbox_inches="tight")
plt.show()

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(cluster_data)
display(df.groupby("cluster")[cluster_features].mean().round(2))
sns.scatterplot(data=df, x="absences", y="G3", hue="cluster", palette="deep")
plt.title("Student clusters: absences and final grade")
plt.tight_layout()
plt.savefig("kmeans_clusters.png", dpi=150, bbox_inches="tight")
plt.show()
print("Clusters are groups with similar scaled values for the selected features. They are exploratory patterns, not proven student types or causal explanations.")

## Problem 14. Model improvement

In [ ]:
# Baseline: a default random forest. Improved model: preprocessing pipeline plus GridSearchCV.
baseline = RandomForestRegressor(random_state=42)
baseline.fit(X_train, y_train)
baseline_predictions = baseline.predict(X_test)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_predictions))

improved_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestRegressor(random_state=42))
])
parameter_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 5, 10],
    "model__min_samples_leaf": [1, 3]
}
search = GridSearchCV(
    improved_pipeline, parameter_grid, cv=5,
    scoring="neg_root_mean_squared_error", n_jobs=-1
)
search.fit(X_train, y_train)
improved_model = search.best_estimator_
improved_predictions = improved_model.predict(X_test)
improved_rmse = np.sqrt(mean_squared_error(y_test, improved_predictions))

improvement_table = pd.DataFrame([
    {"Model": "Baseline Random Forest", "Test RMSE": baseline_rmse},
    {"Model": "Tuned pipeline", "Test RMSE": improved_rmse}
])
display(improvement_table.round(3))
print("Best parameters:", search.best_params_)
print("Cross-validation selected parameters using training data only. The test set was held out for the final comparison.")
print("A lower test RMSE indicates better predictions on this split; cross-validation helps estimate performance across training folds but does not guarantee future performance.")
joblib.dump(improved_model, "student_grade_model.joblib")
print("Saved model: student_grade_model.joblib")

## Problem 15. Reusable prediction program

In [ ]:
# Save the feature names with the fitted pipeline so the separate script uses the same input order.
joblib.dump({
    "model": improved_model,
    "features": regression_features
}, "student_grade_model.joblib")
example = pd.DataFrame([{"studytime": 3, "failures": 0, "absences": 2, "G1": 12, "G2": 14}])
loaded = joblib.load("student_grade_model.joblib")
print("Example input:")
display(example)
print(f"Predicted final grade (G3): {loaded['model'].predict(example[loaded['features']])[0]:.2f} / 20")
print("Run predict_saved_model.py separately to enter your own values.")